# Module 5 – Embeddings, Indexing & Retrieval

## 📍 Where We Are in the Pipeline

```
Document → Extract → Chunk → EMBED → INDEX → RETRIEVE → Generate
                     ✅ M4    🔵 NOW  🔵 NOW  🔵 NOW
```

**This module covers THREE critical pipeline stages:**
1. **🧮 EMBED** – Convert text chunks to 3072-dimensional vectors
2. **📦 INDEX** – Store vectors in Azure AI Search
3. **🔎 RETRIEVE** – Find relevant chunks for user queries

---

## Learning Outcomes

By the end of this module, you will be able to:
- Generate embeddings using `text-embedding-3-large`
- Design index schemas for RAG workloads with vector fields
- Create and populate an Azure AI Search index (Push model)
- Implement text, vector, and hybrid search
- Configure semantic ranking for improved relevance
- Select the right retrieval pattern for different use cases
- **Use Agentic Retrieval for complex multi-part questions (Preview)**

---

## 📋 Dataset Reminder

We're working with **Israel M1 Metro Line** station documents:
- **metro-s36.pdf**: Station 36 (שדרות הציונות) specification
- Contains: Station specs, passenger forecasts, land use tables, maps, figures
- Languages: Hebrew + English

Our queries will focus on Metro station information!

---

## ⏱️ Estimated Time: ~2.5 hours

| Section | Time |
|---------|------|
| Part 0: Setup & Load Chunks | 10 min |
| Part 1: Embeddings | 25 min |
| Part 2: Index Creation | 25 min |
| Part 3: Search Modes | 35 min |
| Part 4: Retrieval Patterns | 35 min |
| Part 5: Agentic Retrieval (Preview) | 30 min |

---

# Part 0: Setup & Load Chunks from Module 4

First, let's load our environment and the chunks we created in Module 4.

In [ ]:
# Cell 0.1: Install/verify dependencies
import sys
!{sys.executable} -m pip install -q azure-search-documents==11.6.0 openai python-dotenv tqdm azure-identity

In [ ]:
# Cell 0.2: Setup and load environment
import os
import sys
import json
from pathlib import Path

# Add src to path for utilities
sys.path.append(str(Path("../../src").resolve()))
from utils import load_env

# Load environment variables
env = load_env()
print("✅ Environment loaded.")

# Set project root for finding files
PROJECT_ROOT = Path("../../").resolve()

# Verify required environment variables
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_SEARCH_ENDPOINT",
]

missing = [v for v in required_vars if not env.get(v)]
if missing:
    raise ValueError(f"❌ Missing environment variables: {missing}")

print(f"   - OpenAI Endpoint: {env.get('AZURE_OPENAI_ENDPOINT', '')[:50]}...")
print(f"   - Search Endpoint: {env.get('AZURE_SEARCH_ENDPOINT', '')[:50]}...")

In [ ]:
# Cell 0.3: Load chunks from Module 4
chunks_path = PROJECT_ROOT / "modules" / "module-4-chunking" / "output" / "hybrid_chunks.json"

if not chunks_path.exists():
    raise FileNotFoundError(f"❌ Chunks file not found: {chunks_path}\n   Please complete Module 4 first.")

with open(chunks_path, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks from Module 4")
print(f"   Source: metro-s36.pdf (Station 36 - שדרות הציונות)")

# Analyze chunk distribution
content_types = {}
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    content_types[ct] = content_types.get(ct, 0) + 1

print(f"\n📊 Chunk Distribution:")
for ct, count in sorted(content_types.items(), key=lambda x: -x[1]):
    print(f"   {ct}: {count}")

In [ ]:
# Cell 0.4: Inspect sample chunks
print("📄 Sample Chunks from Metro Station 36:\n")

# Show one of each type
shown_types = set()
for chunk in chunks:
    ct = chunk.get("content_type", "unknown")
    if ct not in shown_types:
        shown_types.add(ct)
        print(f"--- {ct.upper()} (id: {chunk['id']}) ---")
        content = chunk['content'][:300] + "..." if len(chunk['content']) > 300 else chunk['content']
        print(content)
        print(f"\nMetadata: {chunk.get('metadata', {})}")
        print("\n")
    if len(shown_types) >= 3:
        break

---

# Part 1: Embeddings

## What are Embeddings?

Embeddings are **dense vector representations** of text that capture semantic meaning:
- Similar concepts have vectors that are close together
- `text-embedding-3-large` produces **3072-dimensional** vectors
- Enable **semantic search** beyond keyword matching

### Why Embeddings Matter for Metro Documents

```
"תחנה 36"           →  [0.023, -0.156, 0.089, ..., 0.042]  (3072 floats)
"Station 36"        →  [0.021, -0.152, 0.091, ..., 0.039]  (similar!)
"שדרות הציונות"     →  [0.019, -0.148, 0.092, ..., 0.038]  (also similar!)
"pizza recipe"      →  [-0.234, 0.078, -0.156, ..., -0.089]  (very different)
```

The embedding model understands that "תחנה 36", "Station 36", and "שדרות הציונות" are all related to the same metro station!

## Lab 1.1: Initialize OpenAI Client

In [ ]:
# Cell 1.1: Initialize Azure OpenAI client for embeddings
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Use Entra ID authentication (keys are disabled on this resource)
credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

openai_client = AzureOpenAI(
    azure_endpoint=env["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=token_provider,
    api_version=env.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")
)

EMBEDDING_MODEL = env.get("AZURE_OPENAI_DEPLOYMENT_EMBEDDING", "text-embedding-3-large")
EMBEDDING_DIMENSIONS = 3072

print(f"✅ OpenAI client initialized (using Entra ID)")
print(f"   - Embedding model: {EMBEDDING_MODEL}")
print(f"   - Dimensions: {EMBEDDING_DIMENSIONS}")

## Lab 1.2: Generate a Single Embedding

In [ ]:
# Cell 1.2: Generate embedding for a single text
def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    """
    Generate embedding for a single text.
    
    Args:
        text: Input text (max ~8191 tokens)
        model: Embedding model deployment name
        
    Returns:
        List of floats (3072 dimensions)
    """
    # Clean and truncate text if needed (rough estimate: 4 chars per token)
    max_chars = 8000 * 4  # ~32000 chars
    if len(text) > max_chars:
        text = text[:max_chars]
    
    response = openai_client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

# Test with Metro-related text
test_text = "תחנה מספר 36 שדרות הציונות - קיבולת נוסעים צפויה 2,400 נוסעים בשעת שיא"
test_embedding = get_embedding(test_text)

print(f"✅ Generated embedding for Metro Station 36 text")
print(f"   - Input: {test_text}")
print(f"   - Input length: {len(test_text)} characters")
print(f"   - Output dimensions: {len(test_embedding)}")
print(f"   - First 5 values: {test_embedding[:5]}")

## Lab 1.3: Semantic Similarity Demo

Let's verify that embeddings capture semantic similarity with Metro-related queries:

In [ ]:
# Cell 1.3: Demonstrate semantic similarity with Metro content
import numpy as np

def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    a = np.array(v1)
    b = np.array(v2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test sentences - Metro Station 36 related
sentences = [
    "כמה נוסעים צפויים בתחנה 36?",                          # S1: Hebrew question about passengers
    "How many passengers are expected at Station 36?",       # S2: Same question in English
    "קיבולת נוסעים צפויה 2,400 נוסעים בשעת שיא",            # S3: Answer about passenger capacity
    "I love eating pizza on Friday nights."                  # S4: Completely unrelated
]

print("📊 Semantic Similarity Matrix (Metro Station 36 queries)\n")
print(f"{'':>5}", end="")
for i in range(len(sentences)):
    print(f"  S{i+1}  ", end="")
print("\n")

embeddings_demo = [get_embedding(s) for s in sentences]

for i, emb_i in enumerate(embeddings_demo):
    print(f"S{i+1}  ", end="")
    for j, emb_j in enumerate(embeddings_demo):
        sim = cosine_similarity(emb_i, emb_j)
        print(f" {sim:.3f} ", end="")
    print()

print("\n📝 Sentences:")
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s[:60]}..." if len(s) > 60 else f"S{i+1}: {s}")

print("\n💡 Observations:")
print("   - S1 (Hebrew) and S2 (English) should be highly similar (~0.8+) - same question!")
print("   - S1/S2 and S3 should be related (~0.6+) - question and answer")
print("   - S4 (pizza) should be very different from all (~0.3 or less)")

## Lab 1.4: Batch Embedding Generation

For efficiency, we process multiple texts in batches:

In [ ]:
# Cell 1.4: Batch embedding function
from tqdm import tqdm
import time

def get_embeddings_batch(
    texts: list[str], 
    model: str = EMBEDDING_MODEL, 
    batch_size: int = 16,
    show_progress: bool = True
) -> list[list[float]]:
    """
    Generate embeddings for multiple texts in batches.
    
    Args:
        texts: List of input texts
        model: Embedding model deployment name
        batch_size: Number of texts per API call (max ~16 recommended)
        show_progress: Show progress bar
        
    Returns:
        List of embedding vectors
    """
    all_embeddings = []
    max_chars = 8000 * 4  # Token limit safety
    
    # Process in batches
    batches = [texts[i:i+batch_size] for i in range(0, len(texts), batch_size)]
    
    iterator = tqdm(batches, desc="Generating embeddings") if show_progress else batches
    
    for batch in iterator:
        # Truncate long texts
        batch_cleaned = [t[:max_chars] if len(t) > max_chars else t for t in batch]
        
        try:
            response = openai_client.embeddings.create(
                input=batch_cleaned,
                model=model
            )
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
        except Exception as e:
            print(f"❌ Error in batch: {e}")
            # Add empty embeddings for failed batch (handle gracefully)
            all_embeddings.extend([[0.0] * EMBEDDING_DIMENSIONS] * len(batch))
        
        # Rate limiting - be nice to the API
        time.sleep(0.1)
    
    return all_embeddings

print("✅ Batch embedding function defined")

## Lab 1.5: Generate Embeddings for All Chunks

Now let's embed all our Metro Station 36 chunks from Module 4. This may take a minute.

In [ ]:
# Cell 1.5: Generate embeddings for all chunks

# Extract text content from chunks
chunk_texts = [chunk["content"] for chunk in chunks]

print(f"📊 Embedding {len(chunk_texts)} Metro Station 36 chunks...")
print(f"   - Estimated time: ~{len(chunk_texts) // 16 * 2 + 5} seconds\n")

start_time = time.time()
embeddings = get_embeddings_batch(chunk_texts, batch_size=16)
elapsed = time.time() - start_time

print(f"\n✅ Generated {len(embeddings)} embeddings in {elapsed:.1f}s")
print(f"   - Rate: {len(embeddings)/elapsed:.1f} embeddings/sec")
print(f"   - Each embedding: {len(embeddings[0])} dimensions")

In [ ]:
# Cell 1.6: Attach embeddings to chunks

# Create enriched chunks with embeddings
enriched_chunks = []
for i, chunk in enumerate(chunks):
    enriched_chunk = chunk.copy()
    enriched_chunk["embedding"] = embeddings[i]
    enriched_chunks.append(enriched_chunk)

print(f"✅ Created {len(enriched_chunks)} enriched chunks with embeddings")

# Verify
sample = enriched_chunks[0]
print(f"\n📄 Sample enriched chunk:")
print(f"   - id: {sample['id']}")
print(f"   - content_type: {sample.get('content_type', 'unknown')}")
print(f"   - content length: {len(sample['content'])} chars")
print(f"   - embedding dimensions: {len(sample['embedding'])}")

---

# Part 2: Azure AI Search Index

## Azure AI Search Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Azure AI Search                          │
├─────────────────────────────────────────────────────────────┤
│  ┌──────────────────────────────────────────────────────┐  │
│  │              INDEX: metro-rag-index                   │  │
│  │  ┌────────────────────────────────────────────────┐  │  │
│  │  │  DOCUMENTS (Metro Station 36 chunks)           │  │  │
│  │  │  ┌────┐ ┌────┐ ┌────┐ ┌────┐                   │  │  │
│  │  │  │text│ │text│ │table│ │fig │ ...              │  │  │
│  │  │  └────┘ └────┘ └────┘ └────┘                   │  │  │
│  │  └────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘
```

**Key Concepts:**
- **Index**: Schema definition (like a database table)
- **Document**: Individual item in the index (like a row) - each chunk becomes a document
- **Field**: Attribute of a document (like a column)
- **Vector Field**: Special field type for semantic search

## Lab 2.1: Initialize Search Client

In [ ]:
# Cell 2.1: Initialize Azure AI Search clients
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchableField,
    SimpleField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
)

# Configuration
SEARCH_ENDPOINT = env["AZURE_SEARCH_ENDPOINT"]
SEARCH_API_KEY = env.get("AZURE_SEARCH_API_KEY")  # May be None if using Entra ID
INDEX_NAME = env.get("AZURE_SEARCH_INDEX_NAME", "metro-rag-index")

# Create clients - try API key first, fall back to Entra ID
if SEARCH_API_KEY:
    search_credential = AzureKeyCredential(SEARCH_API_KEY)
    auth_method = "API Key"
else:
    search_credential = credential  # Use Entra ID credential from earlier
    auth_method = "Entra ID"

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential)

print(f"✅ Search clients initialized")
print(f"   - Endpoint: {SEARCH_ENDPOINT}")
print(f"   - Index name: {INDEX_NAME}")
print(f"   - Auth: {auth_method}")

## Lab 2.2: Design the Index Schema

A well-designed schema is critical for RAG performance:

| Field | Type | Purpose |
|-------|------|----------|
| `id` | string | Unique identifier (key) |
| `content` | string | Searchable text content (Hebrew + English) |
| `content_type` | string | Type (text, table, figure) for filtering |
| `embedding` | vector(3072) | Semantic search vector |
| `section_header` | string | Section title for context |
| `strategy` | string | Chunking strategy used |
| `metadata` | string | JSON metadata (flexible) |

In [ ]:
# Cell 2.2: Define the index schema

# Vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="hnsw-config",
            parameters={
                "m": 4,          # Number of bi-directional links (default: 4)
                "efConstruction": 400,  # Size of dynamic list during indexing
                "efSearch": 500,        # Size of dynamic list during search
                "metric": "cosine"      # Distance metric
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="vector-profile",
            algorithm_configuration_name="hnsw-config"
        )
    ]
)

# Semantic search configuration (for L2 reranking)
semantic_config = SemanticConfiguration(
    name="semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
    )
)

semantic_search = SemanticSearch(configurations=[semantic_config])

# Define fields
fields = [
    # Key field (required)
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        filterable=True
    ),
    # Content field (searchable) - supports Hebrew and English
    SearchableField(
        name="content",
        type=SearchFieldDataType.String,
        searchable=True,
        analyzer_name="standard.lucene"  # Works with both Hebrew and English
    ),
    # Content type (for filtering by chunk type)
    SimpleField(
        name="content_type",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Section header (for context)
    SearchableField(
        name="section_header",
        type=SearchFieldDataType.String,
        searchable=True,
        filterable=True
    ),
    # Strategy field
    SimpleField(
        name="strategy",
        type=SearchFieldDataType.String,
        filterable=True,
        facetable=True
    ),
    # Vector embedding field
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="vector-profile"
    ),
    # Metadata as JSON string
    SimpleField(
        name="metadata",
        type=SearchFieldDataType.String,
        filterable=False
    ),
]

print("✅ Index schema defined")
print(f"\n📋 Fields:")
for f in fields:
    print(f"   - {f.name}: {f.type}")

## Lab 2.3: Create the Index

In [ ]:
# Cell 2.3: Create or update the index

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

# Create or update
try:
    result = index_client.create_or_update_index(index)
    print(f"✅ Index '{result.name}' created/updated successfully")
    print(f"   - Vector search: HNSW (cosine similarity)")
    print(f"   - Semantic search: Enabled (for L2 reranking)")
except Exception as e:
    print(f"❌ Error creating index: {e}")
    raise

## Lab 2.4: Upload Documents (Push Model)

Azure AI Search supports two ingestion patterns:
- **Push Model**: Application uploads documents directly via SDK ← We use this
- **Pull Model**: Indexer pulls from data source (Blob, SQL, etc.)

For RAG with pre-computed embeddings, **Push** is typically better.

In [ ]:
# Cell 2.4: Prepare documents for upload

def prepare_document(chunk: dict) -> dict:
    """
    Convert a chunk to a search document.
    
    Args:
        chunk: Enriched chunk with embedding
        
    Returns:
        Document dict ready for indexing
    """
    return {
        "id": chunk["id"],
        "content": chunk["content"],
        "content_type": chunk.get("content_type", "text"),
        "section_header": chunk.get("section_header", ""),
        "strategy": chunk.get("strategy", "unknown"),
        "embedding": chunk["embedding"],
        "metadata": json.dumps(chunk.get("metadata", {}))
    }

# Prepare all documents
documents = [prepare_document(chunk) for chunk in enriched_chunks]

print(f"✅ Prepared {len(documents)} documents for upload")
print(f"\n📄 Sample document:")
sample_doc = documents[0].copy()
sample_doc["embedding"] = f"[{len(sample_doc['embedding'])} floats]"  # Truncate for display
sample_doc["content"] = sample_doc["content"][:100] + "..."
for k, v in sample_doc.items():
    print(f"   {k}: {v}")

In [ ]:
# Cell 2.5: Upload documents in batches

# Create search client for document operations
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=search_credential
)

# Upload in batches of 100 (recommended max)
batch_size = 100
total_uploaded = 0
total_failed = 0

print(f"📤 Uploading {len(documents)} Metro Station 36 chunks in batches of {batch_size}...\n")

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    try:
        result = search_client.upload_documents(documents=batch)
        succeeded = sum(1 for r in result if r.succeeded)
        failed = len(batch) - succeeded
        total_uploaded += succeeded
        total_failed += failed
        print(f"   Batch {i//batch_size + 1}: {succeeded} succeeded, {failed} failed")
    except Exception as e:
        print(f"   Batch {i//batch_size + 1}: ❌ Error - {e}")
        total_failed += len(batch)

print(f"\n✅ Upload complete: {total_uploaded} succeeded, {total_failed} failed")

In [ ]:
# Cell 2.6: Verify index population
import time

# Wait a moment for index to update
time.sleep(2)

# Get document count
results = search_client.search(search_text="*", include_total_count=True)
total_count = results.get_count()

print(f"✅ Index '{INDEX_NAME}' now contains {total_count} documents")

# Get facets by content_type
facet_results = search_client.search(
    search_text="*",
    facets=["content_type"],
    top=0
)
facets = facet_results.get_facets()

if facets and "content_type" in facets:
    print(f"\n📊 Content type distribution:")
    for facet in facets["content_type"]:
        print(f"   - {facet['value']}: {facet['count']}")

---

# Part 3: Search Modes

Azure AI Search supports multiple search modes:

| Mode | How it Works | Best For |
|------|-------------|----------|
| **Text (BM25)** | Keyword matching + TF-IDF | Exact terms, station numbers |
| **Vector** | Cosine similarity on embeddings | Semantic meaning, multilingual |
| **Hybrid** | BM25 + Vector with RRF fusion | General RAG |
| **Semantic** | Hybrid + L2 neural reranking | Production RAG |

## Lab 3.1: Text Search (BM25)

In [ ]:
# Cell 3.1: Text-only search (BM25)
from azure.search.documents.models import QueryType

def search_text(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform text-only (BM25) search.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    results = search_client.search(
        search_text=query,
        query_type=QueryType.SIMPLE,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test text search with Metro query
query = "נוסעים תחנה 36"  # "passengers station 36" in Hebrew
text_results = search_text(query, top=3)

print(f"🔍 Text Search (BM25): '{query}'\n")
for i, r in enumerate(text_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.2: Vector Search (Semantic)

In [ ]:
# Cell 3.2: Vector-only search
from azure.search.documents.models import VectorizedQuery

def search_vector(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform vector-only (semantic) search.
    
    Args:
        query: Search query text (will be embedded)
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=top,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=None,  # No text search
        vector_queries=[vector_query],
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test vector search with natural language query (English!)
# Vector search can find Hebrew content from English query!
query = "How many passengers are expected at the station during peak hours?"
vector_results = search_vector(query, top=3)

print(f"🔍 Vector Search (Semantic): '{query}'\n")
print("💡 Notice: English query finds Hebrew content!\n")
for i, r in enumerate(vector_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.3: Hybrid Search (Text + Vector)

**Hybrid search** combines BM25 and vector search using **Reciprocal Rank Fusion (RRF)**:

```
RRF_score = 1/(k + rank_bm25) + 1/(k + rank_vector)
```

This gives the best of both worlds:
- Exact keyword matching from BM25 ("תחנה 36" matches literally)
- Semantic understanding from vectors ("station" matches "תחנה")

In [ ]:
# Cell 3.3: Hybrid search (text + vector)

def search_hybrid(query: str, top: int = 5, filter: str = None) -> list[dict]:
    """
    Perform hybrid search (BM25 + vector with RRF fusion).
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        
    Returns:
        List of search results with score
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,  # Over-fetch for RRF
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,        # Text search
        vector_queries=[vector_query],  # Vector search
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    return [
        {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"]
        }
        for r in results
    ]

# Test hybrid search with mixed Hebrew/English query
query = "קיבולת נוסעים passengers capacity station 36"
hybrid_results = search_hybrid(query, top=3)

print(f"🔍 Hybrid Search (BM25 + Vector): '{query}'\n")
for i, r in enumerate(hybrid_results, 1):
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}")
    print(f"   Section: {r['section_header']}")
    print(f"   {r['content'][:150]}...\n")

## Lab 3.4: Semantic Ranking (L2 Reranker)

### What is Semantic Ranking?

**Semantic ranking** (also called **L2 reranking**) is a two-stage retrieval process:

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Stage 1 (L1): Hybrid Search                                            │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Query → BM25 + Vector → RRF Fusion → Top 50 candidates         │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                              ↓                                          │
│  Stage 2 (L2): Semantic Reranker                                        │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Transformer model scores each candidate for relevance          │   │
│  │  Returns reranker_score (0-4 scale) + final top K               │   │
│  └─────────────────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────────────────┘
```

### Reranker Score (0-4 Scale)

| Score | Meaning |
|-------|---------|------------|
| 0 | Not relevant at all |
| 1 | Slightly relevant |
| 2 | Moderately relevant |
| 3 | Highly relevant |
| **4** | Perfect match |

> 💡 **Tip**: Filter results with `reranker_score >= 2` for quality answers.

In [ ]:
# Cell 3.4: Hybrid + Semantic ranking
from azure.search.documents.models import QueryType, QueryCaptionType, QueryAnswerType

def search_semantic(
    query: str, 
    top: int = 5, 
    filter: str = None,
    include_answers: bool = True
) -> dict:
    """
    Perform hybrid search with semantic ranking.
    
    Args:
        query: Search query text
        top: Number of results
        filter: OData filter expression
        include_answers: Extract semantic answers
        
    Returns:
        Dict with results and optional answers
    """
    # Get embedding for query
    query_embedding = get_embedding(query)
    
    vector_query = VectorizedQuery(
        vector=query_embedding,
        k_nearest_neighbors=50,
        fields="embedding"
    )
    
    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        query_type=QueryType.SEMANTIC,
        semantic_configuration_name="semantic-config",
        query_caption=QueryCaptionType.EXTRACTIVE,
        query_answer=QueryAnswerType.EXTRACTIVE if include_answers else None,
        top=top,
        filter=filter,
        select=["id", "content", "content_type", "section_header"]
    )
    
    # Extract results
    output = {
        "results": [],
        "answers": []
    }
    
    # Get answers (if available)
    try:
        answers = results.get_answers()
        if answers:
            output["answers"] = [
                {
                    "text": a.text,
                    "score": a.score
                }
                for a in answers
            ]
    except:
        pass
    
    # Get documents
    for r in results:
        doc = {
            "id": r["id"],
            "content": r["content"][:200] + "..." if len(r["content"]) > 200 else r["content"],
            "content_type": r["content_type"],
            "section_header": r.get("section_header", ""),
            "score": r["@search.score"],
            "reranker_score": r.get("@search.reranker_score", None)
        }
        
        # Get captions
        captions = r.get("@search.captions", [])
        if captions:
            doc["caption"] = captions[0].text if hasattr(captions[0], "text") else str(captions[0])
        
        output["results"].append(doc)
    
    return output

# Test semantic search with Metro question
query = "כמה נוסעים צפויים בשעת שיא בתחנה 36?"  # "How many passengers expected at peak hour at Station 36?"
semantic_results = search_semantic(query, top=3)

print(f"🔍 Semantic Search: '{query}'\n")

# Show answers (if any)
if semantic_results["answers"]:
    print("📝 Extracted Answers:")
    for a in semantic_results["answers"]:
        print(f"   Score {a['score']:.2f}: {a['text']}")
    print()

# Show documents
print("📄 Documents:")
for i, r in enumerate(semantic_results["results"], 1):
    reranker_str = f", Reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
    print(f"{i}. [{r['content_type']}] Score: {r['score']:.4f}{reranker_str}")
    if r.get("caption"):
        print(f"   Caption: {r['caption'][:100]}...")
    print(f"   Content: {r['content'][:100]}...\n")

## Lab 3.5: Compare Search Modes

Let's compare all search modes side by side with Metro-specific queries:

In [ ]:
# Cell 3.5: Side-by-side comparison

def compare_search_modes(query: str, top: int = 3):
    """Compare different search modes for the same query."""
    print(f"="*80)
    print(f"Query: '{query}'")
    print(f"="*80)
    
    # Text search
    text_results = search_text(query, top)
    print(f"\n📖 TEXT (BM25):")
    for i, r in enumerate(text_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Vector search
    vector_results = search_vector(query, top)
    print(f"\n🧮 VECTOR (Cosine):")
    for i, r in enumerate(vector_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Hybrid search
    hybrid_results = search_hybrid(query, top)
    print(f"\n🔀 HYBRID (RRF):")
    for i, r in enumerate(hybrid_results, 1):
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f})")
    
    # Semantic search
    semantic_results = search_semantic(query, top, include_answers=False)
    print(f"\n🧠 SEMANTIC (L2 Reranker):")
    for i, r in enumerate(semantic_results["results"], 1):
        reranker = f", reranker: {r['reranker_score']:.2f}" if r['reranker_score'] else ""
        print(f"   {i}. [{r['content_type']}] {r['id']} (score: {r['score']:.3f}{reranker})")

# Test with different query types
print("\n" + "🔬 TEST 1: Hebrew exact term")
compare_search_modes("קיבולת נוסעים")  # "passenger capacity"

print("\n" + "🔬 TEST 2: English semantic question")
compare_search_modes("What is the expected number of passengers at the metro station?")

print("\n" + "🔬 TEST 3: Mixed language (Hebrew + English)")
compare_search_modes("מיקום התחנה location entrances")

---

# Part 4: Retrieval Patterns for RAG

Different RAG scenarios require different retrieval strategies:

| Pattern | Use Case |
|---------|----------|
| **Multi-Retriever** | Mixed content (text + tables + figures) |
| **Filtered Retrieval** | Content-type specific queries |
| **Multimodal** | Questions about diagrams/maps |

## Lab 4.1: Multi-Retriever Pattern

For Metro documents, retrieve from each content type separately and merge:

In [ ]:
# Cell 4.1: Multi-retriever with content-type awareness

def multi_retriever(
    query: str,
    top_per_type: int = 2,
    content_types: list[str] = ["text", "table", "figure"]
) -> dict:
    """
    Retrieve from each content type separately.
    
    This pattern ensures tables (land use data) and figures (maps)
    aren't drowned out by text chunks in the results.
    
    Args:
        query: Search query
        top_per_type: Results per content type
        content_types: Types to query
        
    Returns:
        Dict with results by content type
    """
    results = {}
    
    for ct in content_types:
        filter_expr = f"content_type eq '{ct}'"
        type_results = search_hybrid(query, top=top_per_type, filter=filter_expr)
        results[ct] = type_results
    
    return results

# Test multi-retriever with Metro query
query = "ייעודי קרקע land use"  # "land use" in Hebrew + English
multi_results = multi_retriever(query, top_per_type=2)

print(f"🔍 Multi-Retriever: '{query}'\n")

for content_type, results in multi_results.items():
    print(f"📁 {content_type.upper()} ({len(results)} results):")
    for r in results:
        print(f"   - {r['id']}: {r['content'][:80]}...")
    print()

## Lab 4.2: Filtered Retrieval

Use filters to narrow search based on user intent:

In [ ]:
# Cell 4.2: Intent-based filtered retrieval

def detect_intent(query: str) -> str:
    """
    Simple intent detection for filtering.
    In production, use an LLM for this.
    """
    query_lower = query.lower()
    
    # Check for table indicators (land use, specifications, etc.)
    table_keywords = ["טבלה", "table", "ייעוד", "land use", "מפרט", "specifications", "נתונים", "data"]
    if any(kw in query_lower for kw in table_keywords):
        return "table"
    
    # Check for figure indicators (map, diagram, image, etc.)
    figure_keywords = ["מפה", "map", "תרשים", "diagram", "תמונה", "image", "figure", "show me", "הראה לי"]
    if any(kw in query_lower for kw in figure_keywords):
        return "figure"
    
    return "all"  # No specific intent


def intent_aware_search(query: str, top: int = 5) -> list[dict]:
    """
    Search with automatic intent detection.
    """
    intent = detect_intent(query)
    
    if intent == "table":
        filter_expr = "content_type eq 'table'"
        print(f"🎯 Detected intent: TABLE")
    elif intent == "figure":
        filter_expr = "content_type eq 'figure'"
        print(f"🎯 Detected intent: FIGURE")
    else:
        filter_expr = None
        print(f"🎯 Detected intent: GENERAL")
    
    return search_hybrid(query, top=top, filter=filter_expr)

# Test with different Metro queries
print("\n" + "="*50)
print("Query: 'מה ייעודי הקרקע באזור התחנה?'")
print("       (What are the land use types near the station?)")
results1 = intent_aware_search("מה ייעודי הקרקע באזור התחנה?")  # Land use query
print(f"Results: {[r['id'] for r in results1]}")

print("\n" + "="*50)
print("Query: 'Show me the map of station entrances'")
results2 = intent_aware_search("Show me the map of station entrances")
print(f"Results: {[r['id'] for r in results2]}")

print("\n" + "="*50)
print("Query: 'כמה כניסות יש לתחנה?'")
print("       (How many entrances does the station have?)")
results3 = intent_aware_search("כמה כניסות יש לתחנה?")  # General question
print(f"Results: {[r['id'] for r in results3]}")

## Lab 4.3: RAG Pipeline Integration

This cell creates the **retrieval** portion of a complete RAG pipeline:

```
User Question → Semantic Search → Top K Chunks → Format Context → LLM Prompt
```

In [ ]:
# Cell 4.3: Complete RAG query pipeline

def rag_query(
    question: str,
    top_k: int = 5,
    use_semantic: bool = True
) -> dict:
    """
    Complete RAG query: retrieve + format context for LLM.
    
    Args:
        question: User's question
        top_k: Number of chunks to retrieve
        use_semantic: Use semantic ranking
        
    Returns:
        Dict with retrieved context and metadata
    """
    # Step 1: Retrieve relevant chunks
    if use_semantic:
        search_result = search_semantic(question, top=top_k)
        chunks = search_result["results"]
    else:
        chunks = search_hybrid(question, top=top_k)
    
    # Step 2: Format context for LLM
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        # Format based on content type
        if chunk["content_type"] == "table":
            context_parts.append(f"[Table {i}]:\n{chunk['content']}")
        elif chunk["content_type"] == "figure":
            context_parts.append(f"[Figure {i} description]:\n{chunk['content']}")
        else:
            section = chunk.get('section_header', '')
            section_label = f" (Section: {section})" if section else ""
            context_parts.append(f"[Text {i}{section_label}]:\n{chunk['content']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Step 3: Create prompt template (supports Hebrew + English)
    prompt = f"""Based on the following context about Metro Station 36 (תחנה 36 - שדרות הציונות), answer the question.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question: {question}

Answer:"""
    
    return {
        "question": question,
        "retrieved_chunks": len(chunks),
        "content_types": [c["content_type"] for c in chunks],
        "context_length": len(context),
        "prompt": prompt,
        "chunks": chunks
    }

# Test RAG query with Metro question
question = "כמה נוסעים צפויים בתחנה 36 בשעת השיא?"  # "How many passengers expected at Station 36 during peak hour?"
rag_result = rag_query(question, top_k=3)

print(f"🤖 RAG Query: '{question}'\n")
print(f"📊 Retrieved: {rag_result['retrieved_chunks']} chunks")
print(f"📁 Content types: {rag_result['content_types']}")
print(f"📏 Context length: {rag_result['context_length']} chars")
print(f"\n{'='*60}")
print("PROMPT (truncated):")
print(f"{'='*60}")
print(rag_result['prompt'][:1500] + "...")

## Lab 4.4: Generate Answer with GPT-4.1

In [ ]:
# Cell 4.4: Complete RAG with answer generation

def ask(question: str, top_k: int = 5) -> str:
    """
    Full RAG pipeline: retrieve → generate answer.
    
    Args:
        question: User's question (Hebrew or English)
        top_k: Number of chunks to retrieve
        
    Returns:
        Generated answer
    """
    # Retrieve context
    rag_result = rag_query(question, top_k=top_k)
    
    # Generate answer
    chat_model = env.get("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1")
    
    response = openai_client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": """You are a helpful assistant for Metro Station 36 (תחנה 36 - שדרות הציונות) in Rishon LeZion, Israel.
Answer questions based on the provided context. You can respond in Hebrew or English, matching the language of the question.
If the context doesn't contain enough information, say so."""
            },
            {
                "role": "user",
                "content": rag_result["prompt"]
            }
        ],
        temperature=0.3,
        max_tokens=500
    )
    
    return response.choices[0].message.content

# Test full RAG pipeline with Metro questions
question = "כמה נוסעים צפויים בתחנה 36 בשעת השיא?"  # Passenger capacity question
print(f"❓ Question: {question}")
print("   (How many passengers expected at Station 36 during peak hour?)")
print("\n⏳ Generating answer...\n")

answer = ask(question)
print(f"💡 Answer:\n{answer}")

In [ ]:
# Cell 4.5: Test more Metro questions

test_questions = [
    ("Where is Station 36 located?", "English question about location"),
    ("מה מוקדי העניין הקרובים לתחנה?", "Hebrew question about nearby attractions"),
    ("What types of land use are planned near the station?", "English question about land use"),
]

for q, description in test_questions:
    print(f"\n{'='*60}")
    print(f"❓ {q}")
    print(f"   ({description})")
    print(f"{'='*60}")
    answer = ask(q, top_k=4)
    print(f"\n💡 {answer}")

---

# Part 5: Agentic Retrieval (Preview)

## What is Agentic Retrieval?

**Agentic Retrieval** is a multi-query pipeline in Azure AI Search designed for complex questions. It uses an LLM to decompose your question into focused subqueries.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     Traditional RAG vs Agentic Retrieval                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  TRADITIONAL RAG:                                                           │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question → Single Query → Search → Top K → LLM → Answer   │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
│  AGENTIC RETRIEVAL:                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐       │
│  │  User Question                                                   │       │
│  │         ↓                                                        │       │
│  │  LLM Query Planning (decompose into focused subqueries)          │       │
│  │         ↓                                                        │       │
│  │  ┌─────────┐  ┌─────────┐  ┌─────────┐                          │       │
│  │  │Subquery1│  │Subquery2│  │Subquery3│  (parallel execution)    │       │
│  │  └────┬────┘  └────┬────┘  └────┬────┘                          │       │
│  │       ↓            ↓            ↓                                │       │
│  │  Semantic Rerank + Merge Results                                 │       │
│  └─────────────────────────────────────────────────────────────────┘       │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

### When to Use Agentic Retrieval

| Scenario | Traditional RAG | Agentic Retrieval |
|----------|-----------------|-------------------|
| Simple questions | ✅ | ❌ (overkill) |
| Multi-part questions | ❌ | ✅ |
| Follow-up questions | ❌ | ✅ (context-aware) |
| Cost-sensitive | ✅ | ❌ (higher cost) |

> ⚠️ **Note**: Agentic Retrieval is in **public preview** and requires **Standard tier** or higher.

## Lab 5.0: Prerequisites Check

Agentic Retrieval requires:
- **Standard tier (S1) or higher** - Basic tier is NOT supported
- **Premium features enabled** (Semantic search)
- **RBAC configured** (for managed identity)

In [ ]:
# Cell 5.0: Check prerequisites for Agentic Retrieval
import subprocess

print("🔍 Checking Agentic Retrieval Prerequisites...\n")

# Extract service name from endpoint
SEARCH_SERVICE_NAME = SEARCH_ENDPOINT.replace("https://", "").split(".")[0]
print(f"   Search Service: {SEARCH_SERVICE_NAME}")

# Try to get service info via Azure CLI
try:
    result = subprocess.run(
        f'az search service show --name "{SEARCH_SERVICE_NAME}" --query "{{sku: sku.name, semanticSearch: semanticSearch}}" -o json 2>/dev/null',
        shell=True, capture_output=True, text=True, timeout=30
    )
    if result.returncode == 0 and result.stdout.strip():
        import json
        info = json.loads(result.stdout)
        sku = info.get('sku', 'unknown')
        semantic = info.get('semanticSearch', 'disabled')
        print(f"   SKU: {sku}")
        print(f"   Semantic Search: {semantic}")
        
        if sku.lower() in ['basic', 'free']:
            print("\n⚠️  WARNING: Agentic Retrieval requires Standard tier or higher.")
            print("   Your service is on Basic/Free tier. Labs 5.1+ will be skipped.")
            print("   To upgrade: Azure Portal → Search Service → Settings → Pricing tier")
            AGENTIC_AVAILABLE = False
        elif semantic == 'disabled' or semantic is None:
            print("\n⚠️  WARNING: Semantic search is not enabled.")
            print("   Enable it: Azure Portal → Search Service → Settings → Premium features")
            AGENTIC_AVAILABLE = False
        else:
            print("\n✅ Prerequisites met! Agentic Retrieval is available.")
            AGENTIC_AVAILABLE = True
    else:
        print("   (Could not verify via Azure CLI - will try anyway)")
        AGENTIC_AVAILABLE = True
except Exception as e:
    print(f"   (Azure CLI check failed: {e})")
    print("   Will attempt Agentic Retrieval - it may fail if prerequisites aren't met.")
    AGENTIC_AVAILABLE = True

In [ ]:
# Cell 5.1: Install preview SDK for agentic retrieval
if AGENTIC_AVAILABLE:
    import sys
    !{sys.executable} -m pip install -q azure-search-documents --pre --force-reinstall
    
    # Check installed version
    import importlib.metadata
    version = importlib.metadata.version('azure-search-documents')
    print(f"✅ Installed azure-search-documents version: {version}")
    print("\n⚠️  IMPORTANT: Restart the kernel if you see import errors in the next cell.")
else:
    print("⏭️  Skipping - Agentic Retrieval not available on this service tier.")

In [ ]:
# Cell 5.2: Create Knowledge Source and Knowledge Base
if AGENTIC_AVAILABLE:
    try:
        from azure.search.documents.indexes.models import (
            SearchIndexKnowledgeSource,
            SearchIndexKnowledgeSourceParameters,
            SearchIndexFieldReference,
            KnowledgeBase,
            KnowledgeSourceReference,
            KnowledgeBaseAzureOpenAIModel,
            AzureOpenAIVectorizerParameters,
            KnowledgeRetrievalLowReasoningEffort,
            KnowledgeRetrievalOutputMode,
        )
        
        KNOWLEDGE_SOURCE_NAME = f"{INDEX_NAME}-source"
        KNOWLEDGE_BASE_NAME = f"{INDEX_NAME}-kb"
        
        # Create knowledge source
        knowledge_source = SearchIndexKnowledgeSource(
            name=KNOWLEDGE_SOURCE_NAME,
            description="Metro Station 36 documents for RAG workshop",
            search_index_parameters=SearchIndexKnowledgeSourceParameters(
                search_index_name=INDEX_NAME,
                source_data_fields=[
                    SearchIndexFieldReference(name="id"),
                    SearchIndexFieldReference(name="content_type"),
                    SearchIndexFieldReference(name="section_header"),
                ]
            ),
        )
        
        index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
        print(f"✅ Knowledge source '{KNOWLEDGE_SOURCE_NAME}' created")
        
        # Create knowledge base
        AZURE_OPENAI_ENDPOINT = env["AZURE_OPENAI_ENDPOINT"]
        GPT_MODEL = env.get("AZURE_OPENAI_DEPLOYMENT_GPT41", "gpt-4.1")
        
        aoai_params = AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT,
            deployment_name=GPT_MODEL,
            model_name=GPT_MODEL,
        )
        
        knowledge_base = KnowledgeBase(
            name=KNOWLEDGE_BASE_NAME,
            models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
            knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)],
            output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        )
        
        index_client.create_or_update_knowledge_base(knowledge_base)
        print(f"✅ Knowledge base '{KNOWLEDGE_BASE_NAME}' created")
        print(f"   - LLM: {GPT_MODEL}")
        
    except ImportError as e:
        print(f"❌ Import error - please restart the kernel and run again: {e}")
        AGENTIC_AVAILABLE = False
    except Exception as e:
        print(f"❌ Error creating knowledge base: {e}")
        if "Forbidden" in str(e) or "403" in str(e):
            print("   This usually means your search service tier doesn't support Agentic Retrieval.")
        AGENTIC_AVAILABLE = False
else:
    print("⏭️  Skipping - Agentic Retrieval not available.")

In [ ]:
# Cell 5.3: Test Agentic Retrieval with complex Metro question
if AGENTIC_AVAILABLE:
    try:
        from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
        from azure.search.documents.knowledgebases.models import (
            KnowledgeBaseRetrievalRequest,
            KnowledgeBaseMessage,
            KnowledgeBaseMessageTextContent,
            SearchIndexKnowledgeSourceParams,
        )
        
        # Create retrieval client
        retrieval_client = KnowledgeBaseRetrievalClient(
            endpoint=SEARCH_ENDPOINT,
            knowledge_base_name=KNOWLEDGE_BASE_NAME,
            credential=search_credential
        )
        
        # Complex multi-part question about Metro Station 36
        complex_question = """
        Tell me about Metro Station 36: 
        1. Where is it located?
        2. How many passengers are expected?
        3. What are the nearby attractions?
        """
        
        retrieval_request = KnowledgeBaseRetrievalRequest(
            messages=[
                KnowledgeBaseMessage(
                    role="user",
                    content=[KnowledgeBaseMessageTextContent(text=complex_question)]
                )
            ],
            knowledge_source_params=[
                SearchIndexKnowledgeSourceParams(
                    knowledge_source_name=KNOWLEDGE_SOURCE_NAME,
                    include_references=True,
                    include_reference_source_data=True,
                    always_query_source=True
                )
            ],
            include_activity=True,
            retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort
        )
        
        print("📤 Sending complex multi-part question...")
        print(f"   Question: {complex_question[:100].strip()}...")
        print("\n⏳ Processing (LLM will decompose into subqueries)...\n")
        
        result = retrieval_client.retrieve(retrieval_request=retrieval_request)
        
        print("✅ Agentic Retrieval complete!")
        print(f"   - References: {len(result.references) if result.references else 0}")
        print(f"   - Activities: {len(result.activity) if result.activity else 0}")
        
        # Show subqueries generated
        if result.activity:
            print("\n🔍 Subqueries generated:")
            for activity in result.activity:
                activity_dict = activity.as_dict()
                if 'search_index_arguments' in activity_dict:
                    search = activity_dict['search_index_arguments'].get('search', '')
                    if search:
                        print(f"   • {search[:80]}..." if len(str(search)) > 80 else f"   • {search}")
        
    except ImportError as e:
        print(f"❌ Import error - please restart the kernel: {e}")
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("⏭️  Skipping - Agentic Retrieval not available.")

---

# Summary

## What We Learned

1. **Embeddings** capture semantic meaning - English queries can find Hebrew content!
2. **Azure AI Search** provides multiple search modes:
   - Text (BM25) - keyword matching
   - Vector (kNN) - semantic similarity
   - Hybrid (RRF) - best of both
   - Semantic (L2 reranker) - production quality
3. **Index design** matters: include `content_type` for filtering
4. **Multi-retriever** patterns balance results across tables, figures, text
5. **Full RAG pipeline**: Embed → Index → Retrieve → Generate
6. **Agentic Retrieval** handles complex multi-part questions

## Key Takeaways

| Component | Recommendation |
|-----------|----------------|
| Embedding Model | `text-embedding-3-large` (3072d) |
| Search Mode | Hybrid + Semantic for production |
| Index Design | Include `content_type` field |
| Retrieval | Multi-retriever for mixed content |
| Top-K | Start with 5, adjust based on results |
| Multilingual | Embeddings handle Hebrew ↔ English |

---

## Next Steps

**Module 6: GraphRAG** - Cross-document reasoning with knowledge graphs

---

## Cleanup (Optional)

Run the cell below to delete the index if you want to start fresh:

In [ ]:
# Cell: Cleanup - Delete index (OPTIONAL)
# Uncomment and run if you want to delete the index

# index_client.delete_index(INDEX_NAME)
# print(f"✅ Index '{INDEX_NAME}' deleted")

# If you created agentic retrieval resources:
# index_client.delete_knowledge_base(KNOWLEDGE_BASE_NAME)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE_NAME)
# print(f"✅ Knowledge base and source deleted")

print("💡 Uncomment the code above to delete resources.")